In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import os

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import torch.optim as optimizer

import seaborn as sns

In [2]:
train_val = pd.read_csv('data/train_values.csv') #The data values of each building
train_labels = pd.read_csv('data/train_labels.csv') #labels of data as damage severity ('damage_grade')

test_val = pd.read_csv('data/test_values.csv') #data values of contest submition (our AI would have to guess this, 
#but we won't know their 'damage_grade' for sure until DrivenData releases the answers.

#We combine the features and target label for data sampling and other transformations
df = pd.merge(train_val, train_labels, on='building_id')

#with neural networks, we need damage grade to be 0, 1, 2, instead of 1, 2, 3
#I need to remember to correct for this when submitting
df['damage_grade'] = df['damage_grade'] - 1


In [3]:
columnList = df.drop(['damage_grade', 'building_id'], axis = 1).columns.tolist()

duplicates = df[df.duplicated(subset=columnList, keep=False)]

mostCommon = (
    duplicates.groupby(columnList)['damage_grade']
      .agg(lambda x: x.value_counts().idxmax())
      .reset_index()
)

mostCommon.sort_values(by=columnList).head(10)

#remove duplicates
# df = df.drop_duplicates(subset=columnList, keep=False, inplace=False)
# df.info()


#add back the most common without adding back the outliers
# df2 = pd.concat([df, mostCommon]) 
df2 = df.copy()
print(duplicates.shape)
print(mostCommon.shape)
print(df2.shape)
df2 = df2.fillna(0)
print(df.shape)

(28544, 40)
(12353, 39)
(260601, 40)
(260601, 40)


In [4]:
# df2.info()

In [5]:
#This will state whether neural network is set up to use GPU
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [6]:
#Sampling the data so test runs don't take so long
# dataSample = df2.sample(frac=1)
dataSample = df2.copy()
# dataSample = pd.concat([df2.copy(), df2.copy()])
print(dataSample.shape)
#define X (data features) and y (target)
#X is all useful features
#y is target (final column 'default payment next month')

#Default Features to drop:
#building_id: for recordkeeping rather than data
#The target variable: damage_grade
features = dataSample.drop(['building_id', 'damage_grade'], axis = 1)

print(features.shape, ' features')
#Other features to drop:
# dropColumns = ['geo_level_1_id', 'geo_level_2_id', 'geo_level_3_id'] #Too specific categorical, would produce thousands of columns
#I decided we keep geo_level_1_id as there are only 30 and it is overal region, may have more value
#but we need to make it categorical to generate one-hot encodings
# features['geo_level_1_id'] = features['geo_level_1_id'].astype('category')
# dropColumns = 
secondaryUseTables = [
    'has_secondary_use',
    'has_secondary_use_agriculture',
    'has_secondary_use_hotel',
    'has_secondary_use_rental',
    'has_secondary_use_institution',
    'has_secondary_use_school',
    'has_secondary_use_industry',
    'has_secondary_use_health_post',
    'has_secondary_use_gov_office',
    'has_secondary_use_use_police',
    'has_secondary_use_other',
]

# features = features.drop(secondaryUseTables, axis = 1)
print(features.shape, ' features')

#Data adjustment
# encoder = OneHotEncoder(sparse_output=False, drop='first')

#These tables are in string type and must be converted
# categoryTables = ['geo_level_1_id','land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration', 'legal_ownership_status']
categoryTables = ['land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration', 'legal_ownership_status']

# print(categoryTables.shape)
one_hot_tables = pd.get_dummies(features[categoryTables], dtype=int, drop_first=True)
# one_hot_tables = pd.get_dummies(features[categoryTables], dtype=int)

# encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
# one_hot_tables = pd.DataFrame(encoder.fit_transform(features[categoryTables]))


print(one_hot_tables.shape)
print('oh ^')


#Now we replace our string tables with the encoded tables
X = features.drop(categoryTables, axis = 1)
print(X.shape)
print('last sane one ^')

# X['a'] = one_hot_tables['land_surface_condition_o']
# X['b'] = one_hot_tables['land_surface_condition_t']
X = pd.concat([X, one_hot_tables], axis = 1)
# X = one_hot_tables.join(X, how='left')

# X = X.join(dataSample['damage_grade'], how='left')
# X = pd.concat([X, one_hot_tables]) 
print(X.shape)
X.head(3)

(260601, 40)
(260601, 38)  features
(260601, 38)  features
(260601, 30)
oh ^
(260601, 30)
last sane one ^
(260601, 60)


,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,has_superstructure_adobe_mud,has_superstructure_mud_mortar_stone,has_superstructure_stone_flag,has_superstructure_cement_mortar_stone,has_superstructure_mud_mortar_brick,has_superstructure_cement_mortar_brick,has_superstructure_timber,has_superstructure_bamboo,has_superstructure_rc_non_engineered,has_superstructure_rc_engineered,has_superstructure_other,count_families,has_secondary_use,has_secondary_use_agriculture,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other,land_surface_condition_o,land_surface_condition_t,foundation_type_i,foundation_type_r,foundation_type_u,foundation_type_w,roof_type_q,roof_type_x,ground_floor_type_m,ground_floor_type_v,ground_floor_type_x,ground_floor_type_z,other_floor_type_q,other_floor_type_s,other_floor_type_x,position_o,position_s,position_t,plan_configuration_c,plan_configuration_d,plan_configuration_f,plan_configuration_m,plan_configuration_n,plan_configuration_o,plan_configuration_q,plan_configuration_s,plan_configuration_u,legal_ownership_status_r,legal_ownership_status_v,legal_ownership_status_w
0,6,487,12198,2,30,6,5,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0
1,8,900,2812,2,10,8,7,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0
2,21,363,8973,2,10,5,5,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0


In [7]:
# scaler = MinMaxScaler()
scaler = StandardScaler()
X = scaler.fit_transform(X)
print(X.shape)

y = dataSample['damage_grade']
print(y.shape)

(260601, 60)
(260601,)


In [8]:
# X.head(3)

In [9]:
#splitting data and 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
y_train

185461    2
190537    2
232781    1
213004    1
177786    1
         ..
99791     1
150155    1
112797    2
225588    2
93624     1
Name: damage_grade, Length: 208480, dtype: int64

In [11]:
# X_train.size(0)

In [12]:


X_train_tensor = torch.tensor(X_train)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.long)

X_test_tensor = torch.tensor(X_test)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)

In [13]:
print(X_train_tensor)
print(X_train_tensor.size())

tensor([[ 0.8837,  0.5668, -0.0803,  ..., -0.0754,  0.1962, -0.1019],
        [-1.3568,  1.2792, -1.4148,  ..., -0.0754,  0.1962, -0.1019],
        [ 0.8837,  1.2549, -0.9744,  ..., -0.0754, -5.0962,  9.8157],
        ...,
        [ 1.6306, -0.3709, -1.4093,  ..., -0.0754,  0.1962, -0.1019],
        [ 0.3858,  0.7970, -1.5500,  ..., -0.0754,  0.1962, -0.1019],
        [-1.7303, -0.2279, -1.2077,  ..., -0.0754,  0.1962, -0.1019]],
       dtype=torch.float64)
torch.Size([208480, 60])


In [14]:
print

<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [15]:
#data loader loads data (shocking!)
train_loader = DataLoader(dataset=X_train_tensor, batch_size=40)

In [16]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        #define size of out input (feature number)
        inputSize = X_train_tensor.size(1)
        
        self.flatten = nn.Flatten()
        
        self.linear_relu_stack = nn.Sequential(#The sequential structure of the earthquake classifier
            
            nn.Linear(inputSize, 30), #Bias is on by default if not explicitly set off
            nn.ReLU(),
            nn.Linear(30, 3),
            # nn.ReLU(),
            # nn.Linear(9, 3),
            
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
    
    # def classifier(self,z):
    #     return nnf.softmax(self(z))

    def parameters(self):
        return self.linear_relu_stack.parameters()

In [17]:
model = NeuralNetwork().double()

In [18]:
criterion = nn.CrossEntropyLoss() #this handles softmax for us so we don't need it in our layers

In [19]:
adamOptimizer = optimizer.Adam(model.parameters(), lr=0.001)

In [20]:
y_train_tensor.size()

torch.Size([208480])

In [21]:
# # Train the model
# num_epochs = 150
# history = {'loss': [], 'val_loss': []}
# #training
# for epoch in range(num_epochs):
#     model.train()
#     adamOptimizer.zero_grad()
#     outputs = model(X_train_tensor)
#     loss = criterion(outputs, y_train_tensor)  
#     loss.backward()
#     adamOptimizer.step()
#     history['loss'].append(loss.item())
#     #testing
#     model.eval()
#     with torch.no_grad():
#         outputs_val = model(X_test_tensor)
#         val_loss = criterion(outputs_val, y_test_tensor)  
#         history['val_loss'].append(val_loss.item())

#     if (epoch+1) % 10 == 0:
#         print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Validation Loss: {val_loss.item():.4f}')

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split

# ---- Create Dummy DataFrame (60 features + 1 target) ----




# ---- Stratified Train/Test Split ----
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=44, stratify=y)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)
# num_samples = 500

# ---- Custom Dataset ----
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ---- Create datasets ----
train_dataset = TabularDataset(X_train, y_train.to_numpy())
test_dataset = TabularDataset(X_test, y_test.to_numpy())

# ---- DataLoaders ----
#Dataloaders are iterables over the dataset. So when you iterate over it, 
#it will return B randomly from the dataset collected samples (including the data-sample and the target/label), where B is the batch-size.
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ---- Model (60 -> 30 -> 3) ----
model = nn.Sequential(
    # nn.Linear(60, 30),
    # nn.ReLU(),
    # nn.Linear(30, 3)
    
    # nn.Linear(60, 40),
    # nn.ReLU(),
    # nn.Linear(40, 25),
    # nn.ReLU(),
    # nn.Linear(25, 15),
    # nn.ReLU(),
    # nn.Linear(15, 10),
    # nn.ReLU(),
    # nn.Linear(10, 3)

    # nn.Linear(49, 40),
    # nn.SELU(),
    # nn.Linear(40, 25),
    # nn.SELU(),
    # nn.Linear(25, 15),
    # nn.SELU(),
    # nn.Linear(15, 10),
    # nn.SELU(),
    # nn.Linear(10, 3)

    # nn.Linear(60, 50), #current best with random state 42
    # nn.SELU(),
    # nn.Linear(50, 40),
    # nn.SELU(),
    # nn.Linear(40, 40),
    # nn.SELU(),
    # nn.Linear(40, 25),
    # nn.SELU(),
    # nn.Linear(25, 15),
    # nn.SELU(),
    # nn.Linear(15, 10),
    # nn.SELU(),
    # nn.Linear(10, 3)

    nn.Linear(60, 70),
    nn.SELU(),
    nn.Linear(70, 60),
    nn.SELU(),
    nn.Linear(60, 50),
    nn.SELU(),
    nn.Linear(50, 40),
    nn.SELU(),
    nn.Linear(40, 25),
    nn.SELU(),
    nn.Linear(25, 20),
    nn.SELU(),
    nn.Linear(20, 20),
    nn.SELU(),
    nn.Linear(20, 10),
    nn.SELU(),
    nn.Linear(10, 3)

    
    
)

# ---- Loss and Optimizer ----
# imbalancedWeights = torch.tensor([0.1, 0.6, 0.3]) 
# criterion = nn.CrossEntropyLoss(weight=imbalancedWeights)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ---- Training Loop ----
num_epochs = 800
totalInstancesTrained = 0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_loader:
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        totalInstancesTrained += batch_size

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss:.4f}")

# ---- Testing Loop ----
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y in train_loader:
        outputs = model(batch_x)
        _, predicted = torch.max(outputs, 1)

        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

accuracy = correct / total
print(f"Train Accuracy: {accuracy:.4f}")

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model(batch_x)
        _, predicted = torch.max(outputs, 1)

        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")

(208480, 60)
(52121, 60)
Epoch 1/800, Loss: 5022.9322
Epoch 2/800, Loss: 4787.0271
Epoch 3/800, Loss: 4716.8470
Epoch 4/800, Loss: 4673.1472
Epoch 5/800, Loss: 4635.3922
Epoch 6/800, Loss: 4602.8014
Epoch 7/800, Loss: 4568.7621
Epoch 8/800, Loss: 4535.9301
Epoch 9/800, Loss: 4502.4852
Epoch 10/800, Loss: 4474.4603
Epoch 11/800, Loss: 4456.0290
Epoch 12/800, Loss: 4439.4031
Epoch 13/800, Loss: 4421.1395
Epoch 14/800, Loss: 4403.0722
Epoch 15/800, Loss: 4392.7982
Epoch 16/800, Loss: 4378.7191
Epoch 17/800, Loss: 4363.1570
Epoch 18/800, Loss: 4353.3063
Epoch 19/800, Loss: 4346.6402
Epoch 20/800, Loss: 4332.6415
Epoch 21/800, Loss: 4323.6518
Epoch 22/800, Loss: 4314.4736
Epoch 23/800, Loss: 4303.6881
Epoch 24/800, Loss: 4289.6142
Epoch 25/800, Loss: 4283.6335
Epoch 26/800, Loss: 4275.6761
Epoch 27/800, Loss: 4270.3200
Epoch 28/800, Loss: 4262.2321
Epoch 29/800, Loss: 4251.7723
Epoch 30/800, Loss: 4249.0296
Epoch 31/800, Loss: 4239.5630
Epoch 32/800, Loss: 4232.9427
Epoch 33/800, Loss: 4221

In [23]:
print(totalInstancesTrained)

104240000
